# Machine Learning-Based Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence

**Dataset note:** This is synthetic practice data, not actual Parcl customer data. Results are illustrative only.

Run all cells from the repository root.

In [ ]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

df = pd.read_csv('real_estate_buyer_synthetic_dataset.csv')
print('Raw dataset shape:', df.shape)
display(df.head())

In [ ]:
df = df.drop_duplicates().drop_duplicates(subset=['client_id'], keep='first').copy()
df = df.dropna(subset=['region']).copy()
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['age'] = ((pd.Timestamp('2026-11-11') - df['date_of_birth']).dt.days / 365.25).round()
df.loc[~df['age'].between(18,100), 'age'] = np.nan
df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')
print('Final analysis shape:', df.shape)
assert df['client_id'].nunique() == len(df)

In [ ]:
cat = ['client_type','gender','country','region','acquisition_purpose','loan_applied','referral_channel']
num = ['age','satisfaction_score']
prep = ColumnTransformer([
 ('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num),
 ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]), cat)
])
X = prep.fit_transform(df[cat+num])
rows=[]
for k in range(2,8):
 model=KMeans(n_clusters=k,random_state=42,n_init=10)
 labels=model.fit_predict(X)
 rows.append({'k':k,'inertia':model.inertia_,'silhouette':silhouette_score(X,labels)})
evaluation=pd.DataFrame(rows)
display(evaluation)
best_k=int(evaluation.loc[evaluation.silhouette.idxmax(),'k'])
df['kmeans_cluster']=KMeans(n_clusters=best_k,random_state=42,n_init=10).fit_predict(X)
dense_X=X.toarray() if hasattr(X,'toarray') else X
df['hierarchical_cluster']=AgglomerativeClustering(n_clusters=best_k).fit_predict(dense_X)
display(df.groupby('kmeans_cluster').agg(buyers=('client_id','count'),median_age=('age','median'),investment_share=('acquisition_purpose',lambda s:(s=='Investment').mean()),loan_share=('loan_applied',lambda s:(s=='Yes').mean()),satisfaction=('satisfaction_score','mean')))

## Interpretation and limitations

The previous verified run reported a best silhouette score of **0.112616**, which indicates weak cluster separation. Treat cluster profiles as exploratory, not as strongly distinct natural market segments. The dataset is synthetic practice data and is not actual Parcl customer data.